# Web Scraping Viva Real

## Importando bibliotecas

In [1]:
import re, time, os
import numpy as np
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.common.action_chains import ActionChains
import pandas as pd

## Web Scraper

In [24]:
# Inicializamos as listas para guardar as informações

link_imovel=[] # nesta lista iremos guardar a url
address=[]     # nesta lista iremos guardar o endereço
neighbor=[]    # nesta lista iremos guardar o bairro
anunciante=[]  # nesta lista iremos guardar o anunciante 
area=[]        # nesta lista iremos guardar a area
tipo=[]        # nesta lista iremos guardar o tipo de imóvel
room=[]        # nesta lista iremos guardar a quantidade de quartos
bath=[]        # nesta lista iremos guardar a quantidade de banheiros
park=[]        # nesta lista iremos guardar a quantidade de vagas de garagem
price=[]       # nesta lista iremos guardar o preço do imóvel

# Ele irá solicitar quantas páginas você deseja coletar
pages_number=int(input('Quantas paginas? '))
# inicializa o tempo de execução
tic = time.time()

# Configure chromedriver
# para executar, é necessário que você baixe o chromedriver e deixe ele na mesma pasta de execução, ou mude o path
chromedriver = "./chromedriver.exe"
service = Service(chromedriver)
options = Options()
options.binary_location = "C:\\Program Files\\Google\\Chrome Beta\\Application\\chrome.exe" # caminho do seu navegador Chrome
driver = webdriver.Chrome(service=service, options=options)
actions = ActionChains(driver)

# Informe o link inicial para inicializar o scraping. Você pode trocar entre diversos filtros.
# Este scraper foi desenvolvido para o filtro de alugueis na cidade de santa maria no rio grande do sul. 
# Pode acontecer das informações mudarem caso novos filtros sejam adicionados.
link = f'https://www.dfimoveis.com.br/venda/df/aguas-claras/apartamento/2-quartos'
# link = f'https://www.vivareal.com.br/aluguel/rio-grande-do-sul/santa-maria/?pagina=1#onde=BR-Rio_Grande_do_Sul-NULL-Santa_Maria'
driver.get(link)

# Criando o loop entre as paginas do site
for page in range(1,pages_number+1):
   
    # Definimos um sleep time para não sobrecarregar o site
    time.sleep(15)
    # coletamos todas as informações da página e transformamos em formato legivel
    data = driver.execute_script("return document.getElementsByTagName('html')[0].innerHTML")
    soup_complete_source = BeautifulSoup(data.encode('utf-8'), 'html.parser')
    
    # identificamos todos os itens de card de imóveis
    soup = soup_complete_source.find(id='resultadoDaBuscaDeImoveis')    
    

    # Web-Scraping
    # para cada elemento no conjunto de cards, colete:
    for line in soup.find_all(class_="imovel-card"):
        # colete o endereço completo e o bairro
        try:
            full_address=line.find(class_="ellipse-text body-medium accent-color bold").text.strip()
            address.append(full_address.replace('\n', '')) #Get all address
            # if full_address[:3]=='Rua' or full_address[:7]=='Avenida' or full_address[:8]=='Travessa' or full_address[:7]=='Alameda':
            #     neighbor_first=full_address.strip().find(',')
            #     neighbor_second=full_address.strip().find(',', neighbor_first)
            #     if neighbor_second!=-1:
            #         neighbor_text=full_address.strip()[neighbor_first+2:neighbor_second]
            #         neighbor.append(neighbor_text) # Guarde na lista todos os bairros
            #     else: # Bairro não encontrado
            #         neighbor_text='-'
            #         neighbor.append(neighbor_text) # Caso o bairro não seja encontrado
            # else:
            #     get_comma=full_address.find(',')
            #     if get_comma!=-1:
            #         neighbor_text=full_address[:get_comma]
            #         neighbor.append(neighbor_text) # Guarde na lista todos os bairros com problema de formatação provenientes do proprio website  
            #     else:
            #         get_hif=full_address.find('-')
            #         neighbor_text=full_address[:get_hif]
            #         neighbor.append(neighbor_text)
                    
            # Coleta o link
            full_link=line.get('href')
            link_imovel.append(full_link)
                    
            # Coleta o anunciante
            full_anunciante=line.find(class_='imovel-anunciante').picture.img.get('alt').title()
            anunciante.append(full_anunciante)
                    
            # Coleta a área  
            full_area=line.find(class_="ellipse-text body-small accent-color bold mobile-ellipse-view").text.strip()
            areas = re.findall(r'\d+', full_area)
            areas = [int(i) for i in areas]
            area.append(np.mean(areas)) #Get apto's area
            
            # # Coleta tipologia
            # full_tipo = line.find(class_='property-card__title js-cardLink js-card-title').text.split()[0]
            # full_tipo=full_tipo.replace(' ','')
            # full_tipo=full_tipo.replace('\n','')
            # tipo.append(full_tipo)

            # Coleta numero de quartos
            full_room=line.find(class_="border-1 py-0 px-2 bg-white body-small rounded-pill").text.strip()
            full_room=full_room.replace(' ','')
            full_room=full_room.replace('\n','')
            full_room=full_room.replace('Quartos','')
            full_room=full_room.replace('Quarto','')
            room.append(full_room) #Get apto's rooms

            # # Coleta numero de banheiros
            # full_bath=line.find(class_="property-card__detail-item property-card__detail-bathroom js-property-detail-bathroom").text.strip()        
            # full_bath=full_bath.replace(' ','')
            # full_bath=full_bath.replace('\n','')
            # full_bath=full_bath.replace('Banheiros','')
            # full_bath=full_bath.replace('Banheiro','')
            # bath.append(full_bath) #Get apto's Bathrooms

            # # Coleta numero de vagas de garagem
            # full_park=line.find(class_="property-card__detail-item property-card__detail-garage js-property-detail-garages").text.strip()        
            # full_park=full_park.replace(' ','')
            # full_park=full_park.replace('\n','')
            # full_park=full_park.replace('Vagas','')
            # full_park=full_park.replace('Vaga','')
            # park.append(full_park) #Get apto's parking lot

            # Coleta preço
            full_price=re.sub('[^0-9]','',line.css.select(".imovel-price > h4 > span")[0].text.strip())
            price.append(full_price) #Get apto's parking lot

        except:
            continue
    
    # condicional para parar de iterar entre pages
    
    if page < pages_number:
        receita = driver.find_element(By.XPATH, f'/html/body/main/section/div[2]/div[4]/ul/li[19]/span')
        actions.move_to_element(receita).click().perform()
        # receita.click()
            
# fecha o chromedriver
driver.quit()

# cria um dataframe pandas e salva como um arquivo CSV
for i in range(0,len(address)):
    combinacao=[link_imovel[i],address[i],anunciante[i],area[i],price[i]]
    df=pd.DataFrame(combinacao)
    with open('DfImoveis.csv', 'a', encoding='utf-16', newline='') as f:
        df.transpose().to_csv(f, encoding='iso-8859-1', header=False)

# Tempo de execução
toc = time.time()
get_time=round(toc-tic,3)
print('Finished in ' + str(get_time) + ' seconds')
print(str(len(price))+' results!')

Finished in 20.014 seconds
38 results!


In [4]:
imovel_card = soup.find_all(class_="imovel-card")[2]
print(imovel_card)

<a class="imovel-card position-relative d-flex align-items-center align-items-stretch gap-1 gap-md-2 w-100 h-100 bg-neutral overflow-hidden rounded-3 p-0" href="https://www.dfimoveis.com.br/imovel/apartamento-2-quartos-venda-norte-aguas-claras-df-avenida-das-castanheiras-1241090" onclick="atualizarBanners()" target="_blank">
<div class="imovel-image position-relative w-100">
<div class="imovel-image-icons d-flex align-items-center gap-1">
<div class="abs-cam">
<svg class="web-view" fill="currentColor" height="20" style="color: #133769;" viewbox="0 0 16 16" width="20" xmlns="http://www.w3.org/2000/svg">
<path d="M0 5a2 2 0 0 1 2-2h7.5a2 2 0 0 1 1.983 1.738l3.11-1.382A1 1 0 0 1 16 4.269v7.462a1 1 0 0 1-1.406.913l-3.111-1.382A2 2 0 0 1 9.5 13H2a2 2 0 0 1-2-2V5z" fill-rule="evenodd"></path>
</svg>
<svg class="mobile-view" fill="currentColor" height="12" style="color: #133769;" viewbox="0 0 16 16" width="12" xmlns="http://www.w3.org/2000/svg">
<path d="M0 5a2 2 0 0 1 2-2h7.5a2 2 0 0 1 1.983

In [5]:
full_address=imovel_card.find(class_="ellipse-text body-medium accent-color bold").text.strip()

In [6]:
print(full_address)

Avenida das Castanheiras, NORTE, AGUAS CLARAS


In [7]:
print(imovel_card.get('href'))

https://www.dfimoveis.com.br/imovel/apartamento-2-quartos-venda-norte-aguas-claras-df-avenida-das-castanheiras-1241090


In [8]:
print(imovel_card.find(class_='imovel-anunciante').picture.img.get('alt').title())

Thaís Imobiliaria


In [9]:
print(imovel_card.find(class_="border-1 py-0 px-2 bg-white body-small rounded-pill").text.strip())

2 Quartos


In [10]:
print(re.sub('[^0-9]','',imovel_card.css.select(".imovel-price > h4 > span")[0].text.strip()))

610000


In [14]:
full_area = imovel_card.find(class_="ellipse-text body-small accent-color bold mobile-ellipse-view").text.strip()

In [21]:
areas = re.findall(r'\d+', full_area)
areas = [int(i) for i in areas]
print(areas)

[58]


In [23]:
print(np.mean(areas))

58.0
